In [1]:
"""
Solutions to the end-of-chapter exercises (Géron, Hands-On ML, Ch.9 — Unsupervised Learning)

Dataset: sklearn.datasets.fetch_olivetti_faces() — 400 grayscale 64x64 face
images (already scaled to [0,1]), 40 people, 10 images each.

Exercise 10: Stratified train/valid/test split, cluster with k-means,
             pick a good k, visualize clusters.
Exercise 11: Train a classifier on raw pixels (baseline), then use k-means
             as a dimensionality-reduction step, compare performance, then
             try appending k-means features to the original ones.
Exercise 12: Train a Gaussian Mixture Model (on PCA-reduced faces), sample
             new faces, detect anomalies via score_samples() on modified
             images.
Exercise 13: Use PCA reconstruction error directly as an anomaly score.
"""


'\nSolutions to the end-of-chapter exercises (Géron, Hands-On ML, Ch.9 — Unsupervised Learning)\n\nDataset: sklearn.datasets.fetch_olivetti_faces() — 400 grayscale 64x64 face\nimages (already scaled to [0,1]), 40 people, 10 images each.\n\nExercise 10: Stratified train/valid/test split, cluster with k-means,\n             pick a good k, visualize clusters.\nExercise 11: Train a classifier on raw pixels (baseline), then use k-means\n             as a dimensionality-reduction step, compare performance, then\n             try appending k-means features to the original ones.\nExercise 12: Train a Gaussian Mixture Model (on PCA-reduced faces), sample\n             new faces, detect anomalies via score_samples() on modified\n             images.\nExercise 13: Use PCA reconstruction error directly as an anomaly score.\n'

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_olivetti_faces
from sklearn.model_selection import StratifiedShuffleSplit, GridSearchCV
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.svm import SVC
from sklearn.metrics import silhouette_score, accuracy_score



/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [3]:
# ---------------------------------------------------------------------------
# 0. Load + stratified split (same number of images per person in each set)
# ---------------------------------------------------------------------------
def load_and_split_olivetti(random_state=42):
    faces = fetch_olivetti_faces()
    X, y = faces["data"], faces["target"]  # X already scaled to [0,1]

    # First split off the test set, then split remainder into train/valid.
    # StratifiedShuffleSplit ensures each of the 40 people is represented
    # proportionally in every split (10 images/person means a plain random
    # split could easily leave some person entirely out of one set).
    strat_split_1 = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=random_state)
    train_valid_idx, test_idx = next(strat_split_1.split(X, y))
    X_train_valid, y_train_valid = X[train_valid_idx], y[train_valid_idx]
    X_test, y_test = X[test_idx], y[test_idx]

    strat_split_2 = StratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=random_state)
    train_idx, valid_idx = next(strat_split_2.split(X_train_valid, y_train_valid))
    X_train, y_train = X_train_valid[train_idx], y_train_valid[train_idx]
    X_valid, y_valid = X_train_valid[valid_idx], y_train_valid[valid_idx]

    print(f"Ex10 - split sizes: train={len(X_train)}, valid={len(X_valid)}, "
          f"test={len(X_test)}")
    return X_train, y_train, X_valid, y_valid, X_test, y_test




In [4]:
# ---------------------------------------------------------------------------
# 10. Cluster faces with k-means, pick a good k, visualize
# ---------------------------------------------------------------------------
def exercise_10(X_train, k_range=range(5, 160, 5), output_dir="."):
    # Use silhouette score to pick a good number of clusters — no ground
    # truth labels are used here, this is purely unsupervised model selection.
    silhouette_scores = []
    for k in k_range:
        kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
        labels = kmeans.fit_predict(X_train)
        score = silhouette_score(X_train, labels)
        silhouette_scores.append(score)
        print(f"Ex10 - k={k}: silhouette={score:.4f}")

    best_k = list(k_range)[int(np.argmax(silhouette_scores))]
    print("Ex10 - Best k by silhouette score:", best_k)

    plt.figure(figsize=(8, 5))
    plt.plot(list(k_range), silhouette_scores, "bo-")
    plt.axvline(best_k, color="r", linestyle="--", label=f"best k={best_k}")
    plt.xlabel("Number of clusters (k)")
    plt.ylabel("Silhouette score")
    plt.title("Ex10 - Silhouette score vs k")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{output_dir}/ex10_silhouette.png", dpi=120)
    plt.close()

    best_kmeans = KMeans(n_clusters=best_k, n_init=10, random_state=42)
    cluster_labels = best_kmeans.fit_predict(X_train)

    # Visualize a handful of clusters — show a few face thumbnails per
    # cluster to eyeball whether similar-looking faces group together.
    n_clusters_to_show = min(5, best_k)
    n_faces_per_cluster = 5
    fig, axes = plt.subplots(n_clusters_to_show, n_faces_per_cluster,
                              figsize=(n_faces_per_cluster * 1.5, n_clusters_to_show * 1.5))
    for row, cluster_id in enumerate(range(n_clusters_to_show)):
        member_idx = np.where(cluster_labels == cluster_id)[0][:n_faces_per_cluster]
        for col in range(n_faces_per_cluster):
            ax = axes[row, col] if n_clusters_to_show > 1 else axes[col]
            if col < len(member_idx):
                ax.imshow(X_train[member_idx[col]].reshape(64, 64), cmap="gray")
            ax.axis("off")
        axes[row, 0].set_ylabel(f"Cluster {cluster_id}", rotation=0,
                                labelpad=40, fontsize=9)
    plt.suptitle("Ex10 - Sample faces per cluster")
    plt.tight_layout()
    plt.savefig(f"{output_dir}/ex10_clusters.png", dpi=120)
    plt.close()
    print(f"Ex10 - Saved cluster visualization to {output_dir}/ex10_clusters.png")
    # In practice with Olivetti faces you'll often see clusters that mix
    # multiple people together (k-means on raw pixels is sensitive to pose/
    # lighting more than identity), though some clusters do isolate a
    # single person's photos if the pose is fairly consistent.
    return best_k, best_kmeans, cluster_labels



In [5]:
# ---------------------------------------------------------------------------
# 11. k-means as dimensionality reduction for a classifier
# ---------------------------------------------------------------------------
def exercise_11(X_train, y_train, X_valid, y_valid, random_state=42):
    # --- Baseline: classifier directly on raw pixels ---
    baseline_clf = SVC(kernel="rbf", gamma="scale", random_state=random_state)
    baseline_clf.fit(X_train, y_train)
    baseline_acc = accuracy_score(y_valid, baseline_clf.predict(X_valid))
    print("Ex11 - Baseline (raw pixels) validation accuracy:", baseline_acc)

    # --- k-means as dimensionality reduction: replace each image with its
    # distances to every cluster centroid, then classify on THAT ---
    best_k, best_acc = None, -1
    for k in range(20, 220, 20):
        kmeans = KMeans(n_clusters=k, n_init=10, random_state=random_state)
        X_train_reduced = kmeans.fit_transform(X_train)  # distances to centroids
        X_valid_reduced = kmeans.transform(X_valid)

        clf = SVC(kernel="rbf", gamma="scale", random_state=random_state)
        clf.fit(X_train_reduced, y_train)
        acc = accuracy_score(y_valid, clf.predict(X_valid_reduced))
        print(f"Ex11 - k={k}: validation accuracy (k-means features only) = {acc:.4f}")
        if acc > best_acc:
            best_acc, best_k = acc, k

    print(f"Ex11 - Best k-means-only result: k={best_k}, acc={best_acc:.4f}")
    # Typically WORSE than the raw-pixel baseline — reducing a 4096-dim face
    # down to e.g. 100 cluster-distance features throws away too much
    # identity-specific detail for a classification task this fine-grained
    # (telling apart 40 different people).

    # --- Append k-means features to the original pixel features ---
    best_appended_acc = -1
    best_appended_k = None
    for k in [best_k, best_k // 2, best_k * 2]:
        if k < 2:
            continue
        kmeans = KMeans(n_clusters=k, n_init=10, random_state=random_state)
        X_train_dist = kmeans.fit_transform(X_train)
        X_valid_dist = kmeans.transform(X_valid)

        X_train_appended = np.c_[X_train, X_train_dist]
        X_valid_appended = np.c_[X_valid, X_valid_dist]

        clf = SVC(kernel="rbf", gamma="scale", random_state=random_state)
        clf.fit(X_train_appended, y_train)
        acc = accuracy_score(y_valid, clf.predict(X_valid_appended))
        print(f"Ex11 - k={k}: validation accuracy (pixels + k-means features) = {acc:.4f}")
        if acc > best_appended_acc:
            best_appended_acc, best_appended_k = acc, k

    print(f"Ex11 - Best appended-features result: k={best_appended_k}, "
          f"acc={best_appended_acc:.4f}")
    # This usually recovers close to (or matches) the baseline — the
    # k-means features add a little useful signal without discarding the
    # original detail the way pure k-means reduction did.
    return {
        "baseline_acc": baseline_acc,
        "kmeans_only_best": (best_k, best_acc),
        "appended_best": (best_appended_k, best_appended_acc),
    }



In [6]:

# ---------------------------------------------------------------------------
# 12. Gaussian Mixture Model: generate new faces + detect anomalies
# ---------------------------------------------------------------------------
def make_anomalous_faces(X, rng=None):
    """Create a few modified (anomalous) versions of real faces: rotated,
    flipped, and darkened, matching the exercise's suggestions."""
    rng = rng or np.random.default_rng(42)
    n_take = min(4, len(X))
    idx = rng.choice(len(X), size=n_take, replace=False)
    originals = X[idx].reshape(-1, 64, 64)

    rotated = np.rot90(originals, k=1, axes=(1, 2)).reshape(n_take, -1)
    flipped = originals[:, :, ::-1].reshape(n_take, -1)
    darkened = (originals * 0.3).reshape(n_take, -1)

    anomalies = np.vstack([rotated, flipped, darkened])
    return anomalies


def exercise_12(X_train, output_dir=".", random_state=42):
    # Reduce dimensionality first — GMM on raw 4096-dim pixels is slow and
    # the covariance estimation gets unstable with that many dimensions
    # relative to sample count.
    pca = PCA(n_components=0.99, random_state=random_state)
    X_train_pca = pca.fit_transform(X_train)
    print(f"Ex12 - PCA reduced {X_train.shape[1]} -> {X_train_pca.shape[1]} dims "
          f"(99% variance)")

    gm = GaussianMixture(n_components=40, random_state=random_state)
    gm.fit(X_train_pca)
    print("Ex12 - GMM converged:", gm.converged_)

    # Generate new faces
    n_generate = 10
    X_generated_pca, _ = gm.sample(n_generate)
    X_generated = pca.inverse_transform(X_generated_pca)

    fig, axes = plt.subplots(2, 5, figsize=(10, 4))
    for i, ax in enumerate(axes.flat):
        ax.imshow(X_generated[i].reshape(64, 64), cmap="gray")
        ax.axis("off")
    plt.suptitle("Ex12 - Faces generated by the GMM")
    plt.tight_layout()
    plt.savefig(f"{output_dir}/ex12_generated_faces.png", dpi=120)
    plt.close()
    print(f"Ex12 - Saved generated faces to {output_dir}/ex12_generated_faces.png")

    # Anomaly detection: compare score_samples() on normal vs modified faces
    normal_sample = X_train[:10]
    normal_scores = gm.score_samples(pca.transform(normal_sample))

    anomalies = make_anomalous_faces(X_train)
    anomaly_scores = gm.score_samples(pca.transform(anomalies))

    print("Ex12 - Mean log-density, normal faces:   ", normal_scores.mean())
    print("Ex12 - Mean log-density, modified faces: ", anomaly_scores.mean())
    # Expect anomaly_scores to be substantially LOWER (more negative) — the
    # GMM assigns modified/unnatural-looking faces much lower likelihood
    # under the learned face distribution than genuine ones.

    fig, axes = plt.subplots(2, len(anomalies), figsize=(len(anomalies) * 1.5, 3))
    for i in range(len(anomalies)):
        axes[0, i].imshow(anomalies[i].reshape(64, 64), cmap="gray")
        axes[0, i].set_title(f"{anomaly_scores[i]:.0f}", fontsize=8)
        axes[0, i].axis("off")
    for i in range(min(len(anomalies), len(normal_sample))):
        axes[1, i].imshow(normal_sample[i].reshape(64, 64), cmap="gray")
        axes[1, i].set_title(f"{normal_scores[i]:.0f}", fontsize=8)
        axes[1, i].axis("off")
    axes[0, 0].set_ylabel("Modified", fontsize=9)
    axes[1, 0].set_ylabel("Normal", fontsize=9)
    plt.suptitle("Ex12 - GMM log-density scores (higher = more 'normal')")
    plt.tight_layout()
    plt.savefig(f"{output_dir}/ex12_anomaly_scores.png", dpi=120)
    plt.close()

    return gm, pca, normal_scores, anomaly_scores



In [10]:
# ---------------------------------------------------------------------------
# 13. PCA reconstruction error as an anomaly score
# ---------------------------------------------------------------------------
def reconstruction_error(pca, X):
    X_reduced = pca.transform(X)
    X_reconstructed = pca.inverse_transform(X_reduced)
    return np.mean(np.square(X - X_reconstructed), axis=1)


def exercise_13(X_train, output_dir=".", random_state=42):
    pca = PCA(n_components=0.99, random_state=random_state)
    pca.fit(X_train)

    normal_sample = X_train[:10]
    normal_errors = reconstruction_error(pca, normal_sample)

    anomalies = make_anomalous_faces(X_train)
    anomaly_errors = reconstruction_error(pca, anomalies)

    print("Ex13 - Mean reconstruction error, normal faces:  ", normal_errors.mean())
    print("Ex13 - Mean reconstruction error, modified faces:", anomaly_errors.mean())
    # Expect anomaly_errors to be substantially HIGHER — PCA was fit only on
    # normal, well-aligned faces, so it can't reconstruct rotated/flipped/
    # darkened images well; it tries to reconstruct "the nearest normal
    # face it knows," which looks visibly wrong for the modified inputs.

    # Visualize a reconstructed anomalous image next to the original, to
    # see the "it tries to reconstruct a normal face" effect described in
    # the exercise.
    X_anom_reduced = pca.transform(anomalies)
    X_anom_reconstructed = pca.inverse_transform(X_anom_reduced)

    n_show = min(4, len(anomalies))
    fig, axes = plt.subplots(2, n_show, figsize=(n_show * 1.5, 3))
    for i in range(n_show):
        axes[0, i].imshow(anomalies[i].reshape(64, 64), cmap="gray")
        axes[0, i].set_title(f"err={anomaly_errors[i]:.4f}", fontsize=8)
        axes[0, i].axis("off")
        axes[1, i].imshow(X_anom_reconstructed[i].reshape(64, 64), cmap="gray")
        axes[1, i].axis("off")
    axes[0, 0].set_ylabel("Modified\n(input)", fontsize=8)
    axes[1, 0].set_ylabel("PCA\nreconstruction", fontsize=8)
    plt.suptitle("Ex13 - PCA reconstruction of modified (anomalous) faces")
    plt.tight_layout()
    plt.savefig(f"{output_dir}/ex13_reconstruction.png", dpi=120)
    plt.close()
    print(f"Ex13 - Saved reconstruction comparison to {output_dir}/ex13_reconstruction.png")

    return pca, normal_errors, anomaly_errors


if __name__ == "__main__":
    print("Loading Olivetti faces (requires network access)...")
    X_train, y_train, X_valid, y_valid, X_test, y_test = load_and_split_olivetti()

    print("\n=== Exercise 10 ===")
    best_k, best_kmeans, cluster_labels = exercise_10(X_train, k_range=range(20, 100, 20))

    print("\n=== Exercise 11 ===")
    ex11_results = exercise_11(X_train, y_train, X_valid, y_valid)

    print("\n=== Exercise 12 ===")
    gm, pca_gmm, normal_scores, anomaly_scores = exercise_12(X_train)

    print("\n=== Exercise 13 ===")
    pca_recon, normal_errors, anomaly_errors = exercise_13(X_train)


Loading Olivetti faces (requires network access)...
downloading Olivetti faces from https://ndownloader.figshare.com/files/5976027 to /Users/sulabhyadav/scikit_learn_data
Ex10 - split sizes: train=240, valid=80, test=80

=== Exercise 10 ===
Ex10 - k=20: silhouette=0.0985
Ex10 - k=40: silhouette=0.1537
Ex10 - k=60: silhouette=0.1916
Ex10 - k=80: silhouette=0.2001
Ex10 - Best k by silhouette score: 80
Ex10 - Saved cluster visualization to ./ex10_clusters.png

=== Exercise 11 ===
Ex11 - Baseline (raw pixels) validation accuracy: 0.925
Ex11 - k=20: validation accuracy (k-means features only) = 0.5750
Ex11 - k=40: validation accuracy (k-means features only) = 0.6625
Ex11 - k=60: validation accuracy (k-means features only) = 0.6875
Ex11 - k=80: validation accuracy (k-means features only) = 0.6875
Ex11 - k=100: validation accuracy (k-means features only) = 0.7500
Ex11 - k=120: validation accuracy (k-means features only) = 0.7000
Ex11 - k=140: validation accuracy (k-means features only) = 0.71